# Trabajo Práctico Integrador – Entrega 2

**Materia:** Introducción al Análisis de Datos  
**Tema:** Cancelación de reservas hoteleras  
**Comisión:** 10 (Grupo J)  
**Integrante:** Nicolás Viruel  
**Entrega:** 2 – Unidad N° 2 (inicio Semana 4)  
**Año:** 2026

> Notebook **independiente**: ejecutable desde cero sin depender de la Entrega 1.

## 1. Contexto

Analizamos cancelaciones de reservas hoteleras. **Variable objetivo:** `is_canceled` (0 = no cancelada, 1 = cancelada).

En esta entrega nos enfocamos en **diagnóstico y limpieza inicial** del dataset asignado.

## 2. Carga del dataset

- **Archivo:** `../datos/hotel_booking_TPI_grupo_J.csv`  
- **Comisión:** 10 · Grupo J

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../datos')
archivo = DATA_DIR / 'hotel_booking_TPI_grupo_J.csv'
df_hotel = pd.read_csv(archivo)
print('Dataset cargado:', df_hotel.shape)

Dataset cargado: (25000, 32)


## 3. Diagnóstico y limpieza inicial de datos

### 3.1 Preparativos

Copia de trabajo y bitácora de decisiones (Problema | Variable | Decisión | Justificación).

In [2]:
df_hotel_original = df_hotel.copy()
df = df_hotel.copy()

registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        'problema_detectado': problema,
        'variable': variable,
        'decision': decision,
        'justificacion': justificacion,
    })

### 3.2 Estructura, tipos y dimensiones

In [3]:
print('Dimensiones:', df.shape)
df.info()
print('\nDuplicados exactos:', df.duplicated().sum())

Dimensiones: (25000, 32)
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                           25000 non-null  str    
 2   is_canceled                     25000 non-null  int64  
 3   lead_time                       25000 non-null  int64  
 4   arrival_date_year               25000 non-null  int64  
 5   arrival_date_month              25000 non-null  str    
 6   arrival_date_week_number        25000 non-null  int64  
 7   arrival_date_day_of_month       25000 non-null  int64  
 8   arrival_date                    25000 non-null  str    
 9   stays_in_weekend_nights         25000 non-null  int64  
 10  stays_in_week_nights            25000 non-null  int64  
 11  adults                          25000 non-null  int64  
 12  children          

In [4]:
cols_sugeridas = [
    'children', 'adults', 'babies', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adr', 'lead_time', 'country', 'agent', 'company',
]
df[cols_sugeridas].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
children,25000.0,NaN,NaN,NaN,0.10308,0.393218,0.0,0.0,0.0,0.0,3.0
adults,25000.0,NaN,NaN,NaN,1.85444,0.571371,0.0,2.0,2.0,2.0,40.0
babies,25000.0,NaN,NaN,NaN,0.00804,0.091519,0.0,0.0,0.0,0.0,2.0
stays_in_weekend_nights,25000.0,NaN,NaN,NaN,0.92044,0.998193,0.0,0.0,1.0,2.0,18.0
stays_in_week_nights,25000.0,NaN,NaN,NaN,2.499,1.903613,0.0,1.0,2.0,3.0,42.0
adr,25000.0,NaN,NaN,NaN,101.681678,47.969113,0.0,69.0275,94.5,126.0,387.0
lead_time,25000.0,NaN,NaN,NaN,104.22324,107.098048,0.0,18.0,69.0,160.0,629.0
country,24883,134,PRT,10159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agent,21521.0,NaN,NaN,NaN,86.426746,111.048259,1.0,9.0,14.0,229.0,535.0
company,1440.0,NaN,NaN,NaN,187.325694,128.592996,9.0,62.0,186.0,268.0,541.0


### 3.3 Valores faltantes

In [5]:
nulos = df.isna().sum().sort_values(ascending=False)
print(nulos[nulos > 0])
print('\nCadenas vacías en country:', (df['country'] == '').sum())

company    23560
agent       3479
country      117
dtype: int64

Cadenas vacías en country: 0


In [6]:
registrar(
    'Alta proporción de company nulo',
    'company',
    'Conservar NaN',
    '94% de reservas no tienen empresa; es un patrón real, no un error masivo.',
)
registrar(
    'Agent nulo en ~14% de reservas',
    'agent',
    'Conservar NaN',
    'Reservas directas o sin intermediario; relevante para el análisis.',
)
registrar(
    'Country ausente en 117 reservas',
    'country',
    'Conservar NaN',
    'Son pocas filas (<0.5%); eliminar sesgaría países minoritarios.',
)

### 3.4 Valores imposibles o inconsistentes

In [7]:
print('adr <= 0:', (df['adr'] <= 0).sum())
print('adults == 0:', (df['adults'] == 0).sum())
print('Combinación sin huéspedes (adults+children+babies==0):',
      ((df['adults'] + df['children'] + df['babies']) == 0).sum())
df.loc[df['adr'] <= 0, ['hotel', 'adr', 'adults', 'children', 'babies', 'is_canceled']].head()

adr <= 0: 446
adults == 0: 80
Combinación sin huéspedes (adults+children+babies==0): 37


,hotel,adr,adults,children,babies,is_canceled
34,City Hotel,0.0,1,0.0,0,0
94,Resort Hotel,0.0,1,0.0,0,0
133,City Hotel,0.0,2,0.0,0,1
158,City Hotel,0.0,0,0.0,0,0
175,City Hotel,0.0,1,0.0,0,0


In [8]:
mask_adr_invalido = df['adr'] <= 0
print('Reservas con adr <= 0:', mask_adr_invalido.sum())
registrar(
    'Tarifa diaria adr nula, cero o negativa',
    'adr',
    'Marcar para revisión; conservar por ahora',
    'Puede ser cortesía o error de carga; eliminar 446 filas sin contexto sería agresivo.',
)

mask_sin_adultos = df['adults'] == 0
registrar(
    'Reservas con adults == 0',
    'adults',
    'Conservar y analizar',
    'Podría ser error o reserva especial; requiere contexto de negocio.',
)

mask_huespedes = (df['adults'] + df['children'] + df['babies']) == 0
if mask_huespedes.any():
    registrar(
        'Estadía sin huéspedes registrados',
        'adults, children, babies',
        'Conservar para revisión',
        'Combinación imposible en operación normal; posible error de registro.',
    )

Reservas con adr <= 0: 446


### 3.5 Posibles valores atípicos (IQR)

In [9]:
def detectar_outliers_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lim_inf) | (s > lim_sup)]

for col in ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights']:
    out = detectar_outliers_iqr(df[col])
    print(f'{col}: {len(out)} atípicos IQR (min={df[col].min()}, max={df[col].max()})')

lead_time: 657 atípicos IQR (min=0, max=629)
adr: 777 atípicos IQR (min=0.0, max=387.0)
stays_in_week_nights: 679 atípicos IQR (min=0, max=42)
stays_in_weekend_nights: 52 atípicos IQR (min=0, max=18)


In [10]:
registrar(
    'lead_time muy alto (hasta 629 días)',
    'lead_time',
    'Conservar',
    'Anticipación extrema es posible en hoteles resort; no es error automático.',
)
registrar(
    'adr atípico por IQR',
    'adr',
    'Conservar',
    'Tarifas premium existen; se evaluará en EDA si distorsionan modelos.',
)

### 3.6 Variables temporales

In [11]:
df['arrival_date'] = pd.to_datetime(df['arrival_date'], errors='coerce')
fechas_invalidas = df['arrival_date'].isna().sum()
print('Fechas arrival_date inválidas:', fechas_invalidas)
print('Rango:', df['arrival_date'].min(), '→', df['arrival_date'].max())
registrar(
    'Unificación de arrival_date',
    'arrival_date',
    'Convertir a datetime',
    'Permite análisis temporal coherente; no se detectaron fechas inválidas.',
)

Fechas arrival_date inválidas: 0
Rango: 2023-07-01 00:00:00 → 2025-08-31 00:00:00


### 3.7 Bitácora de decisiones

In [12]:
bitacora = pd.DataFrame(registros_bitacora)
bitacora

,problema_detectado,variable,decision,justificacion
0,Alta proporción de company nulo,company,Conservar NaN,94% de reservas no tienen empresa; es un patró...
1,Agent nulo en ~14% de reservas,agent,Conservar NaN,Reservas directas o sin intermediario; relevan...
2,Country ausente en 117 reservas,country,Conservar NaN,Son pocas filas (<0.5%); eliminar sesgaría paí...
3,"Tarifa diaria adr nula, cero o negativa",adr,Marcar para revisión; conservar por ahora,Puede ser cortesía o error de carga; eliminar ...
4,Reservas con adults == 0,adults,Conservar y analizar,Podría ser error o reserva especial; requiere ...
5,Estadía sin huéspedes registrados,"adults, children, babies",Conservar para revisión,Combinación imposible en operación normal; pos...
6,lead_time muy alto (hasta 629 días),lead_time,Conservar,Anticipación extrema es posible en hoteles res...
7,adr atípico por IQR,adr,Conservar,Tarifas premium existen; se evaluará en EDA si...
8,Unificación de arrival_date,arrival_date,Convertir a datetime,Permite análisis temporal coherente; no se det...


### 3.8 Dataset de trabajo

In [13]:
df_hotel_limpio = df.copy()
print('Original:', df_hotel_original.shape, '| Trabajo:', df_hotel_limpio.shape)
df_hotel_limpio[cols_sugeridas].head()

Original: (25000, 32) | Trabajo: (25000, 32)


,children,adults,babies,stays_in_weekend_nights,stays_in_week_nights,adr,lead_time,country,agent,company
0,0.0,2,0,2,5,42.95,261,GBR,40.0,NaN
1,0.0,2,0,0,3,64.00,85,PRT,67.0,NaN
2,0.0,2,0,0,2,101.50,364,PRT,6.0,NaN
3,0.0,1,0,2,0,114.00,32,CHE,9.0,NaN
4,0.0,2,0,0,2,89.10,126,PRT,27.0,NaN


## 4. Cierre parcial (Semana 4)

Se completó el **diagnóstico inicial** del dataset hotelero: estructura revisada, faltantes documentados, valores imposibles identificados (`adr <= 0`, combinaciones de huéspedes) y atípicos analizados con IQR sin eliminación automática.

En la **Semana 5** se continuará con transformaciones adicionales y se entregará formalmente esta Entrega 2.